# 02 · Obtener las carreras reales (HITO 1)

Issue #7. Usa las funciones de `src/carreras.py`, que a su vez piden todas las páginas con
`descargar()` (User-Agent propio, pausa de 1,5 s y sin repetir descargas).

1. **Descubrir las carreras** de la edición actual y la URL de su categoría **Absoluta**.
2. **Descargar todas las páginas** de cada Absoluta en `datos/raw/<carrera_id>/pagina_N.html`.
3. **Tablas por carrera** en `datos/interim/carrera_XX.csv` y **comprobación de filas**.

> En este notebook **nunca se muestran nombres ni dorsales**: solo recuentos.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from src import obtencion
from src.carreras import (descubrir_carreras, descargar_todas, construir_todas,
                          paginas_guardadas, COLUMNAS)
from src.config import DIR_RAW, DIR_INTERIM, DIR_DOCS, FECHA_CORTE

## 1. Descubrir las carreras y su Absoluta

Se parte de la portada de resultados (`/web/resultado/`) y se siguen los enlaces:
**Edición Actual → Por evento → cada carrera → categorías**.

- La Absoluta se localiza por el **texto del enlace** (empieza por «ABSOLUTA»), no por su
  posición ni por el número de competición, que cambian entre carreras.
- La fecha sale de la ficha de cada carrera (`/gl/web/evento/<id>/`).
- Se excluye la carrera que **aún no se ha corrido** (fecha igual o posterior a hoy). Como el
  menú de la web tiene todas las carreras de la edición, también se detecta la última aunque
  todavía no tenga enlace de resultados.

In [2]:
# Fecha de corte fija (ver src/config.py): sólo las carreras anteriores a Novo Mesoiro.
carreras = descubrir_carreras(hoy=FECHA_CORTE)
carreras[["carrera_id", "orden", "fecha", "evento_id", "nombre", "url_absoluta"]]

Carreras disputadas: 6 · excluidas: 1
  - excluida evento 3216 (Novo Mesoiro, 2026-11-22): aún no se ha corrido


,carrera_id,orden,fecha,evento_id,nombre,url_absoluta
0,carrera_01,1,2026-03-14,3210,XIV CARREIRA POPULAR TORRE DE HÉRCULES -TROFEO...,https://corunacorre.com/gl/web/resultado/compe...
1,carrera_02,2,2026-04-12,3211,X CARREIRA POPULAR MATOGRANDE - XUXÁN,https://corunacorre.com/gl/web/resultado/compe...
2,carrera_03,3,2026-05-10,3212,XVII BAIXADA POPULAR DE CROSS SAN PEDRO DE VISMA,https://corunacorre.com/gl/web/resultado/compe...
3,carrera_04,4,2026-05-31,3213,XIII CARREIRA POPULAR OS ROSALES,https://corunacorre.com/gl/web/resultado/compe...
4,carrera_05,5,2026-09-13,3214,XIII CARREIRA POPULAR VOLTA A OZA,https://corunacorre.com/gl/web/resultado/compe...
5,carrera_06,6,2026-10-25,3215,XII CARREIRA POPULAR VENTORRILLO,https://corunacorre.com/gl/web/resultado/compe...


In [3]:
# Guardamos la tabla de carreras (no tiene datos personales)
carreras.to_csv(DIR_INTERIM / "carreras_descubiertas.csv", index=False, encoding="utf-8")
assert len(carreras) > 0 and carreras["url_absoluta"].notna().all()

## 2. Descargar todas las páginas de cada Absoluta

No se sabe de antemano cuántas páginas tiene cada carrera: se pide `?page=1`, `?page=2`...
hasta que la página no tiene filas o no hay enlace «Seguinte». Cada página se guarda en bytes en
`datos/raw/<carrera_id>/pagina_N.html`; si ya existe, se lee del disco sin hacer petición.

In [4]:
paginas = descargar_todas(carreras)
print("Total:", paginas.sum(), "páginas")

carrera_01: 60 páginas


carrera_02: 38 páginas


carrera_03: 32 páginas


carrera_04: 35 páginas


carrera_05: 27 páginas


carrera_06: 26 páginas
Total: 218 páginas


### Segunda ejecución: ninguna petición nueva

Contamos las peticiones reales sustituyendo temporalmente el `get` de la sesión: si todo está
en disco, el contador queda a 0.

In [5]:
peticiones = 0
get_original = obtencion._sesion.get

def get_contado(*args, **kwargs):
    global peticiones
    peticiones += 1
    return get_original(*args, **kwargs)

obtencion._sesion.get = get_contado
try:
    paginas_2 = descargar_todas(carreras)
finally:
    obtencion._sesion.get = get_original

print("Peticiones nuevas en la segunda ejecución:", peticiones)
assert peticiones == 0 and paginas_2.equals(paginas)

carrera_01: 60 páginas


carrera_02: 38 páginas


carrera_03: 32 páginas


carrera_04: 35 páginas


carrera_05: 27 páginas


carrera_06: 26 páginas
Peticiones nuevas en la segunda ejecución: 0


## 3. Tablas por carrera en `interim/` y comprobación de filas

Cada carrera se pasa a `datos/interim/carrera_XX.csv` con las columnas crudas, tal cual salen
de la web (todo como texto; la limpieza es la issue #9):
`posto, dorsal, nome, apelidos, posto_sexo, posto_categoria, tempo`.

La comprobación (solo recuentos) mira, para cada carrera con *N* páginas de *T* filas:

- que todas las páginas salvo la última tengan *T* filas,
- que el total esté entre (*N*−1)·*T*+1 y *N*·*T*,
- que *N* sea la última página que anuncia la paginación de la web,
- que no haya filas repetidas entre páginas.

Se guarda en `docs/comprobacion_filas.csv` (se puede subir: no tiene datos personales).

In [6]:
comprobacion = construir_todas(carreras)
comprobacion

,carrera_id,orden,fecha,paginas,ultima_pagina_web,filas_por_pagina,filas_ultima_pagina,filas,filas_min_esperadas,filas_max_esperadas,paginas_incompletas,filas_duplicadas,cuadra
0,carrera_01,1,2026-03-14,60,60,25,14,1489,1476,1500,0,0,True
1,carrera_02,2,2026-04-12,38,38,25,4,929,926,950,0,0,True
2,carrera_03,3,2026-05-10,32,32,25,20,795,776,800,0,0,True
3,carrera_04,4,2026-05-31,35,35,25,25,875,851,875,0,0,True
4,carrera_05,5,2026-09-13,27,27,25,9,659,651,675,0,0,True
5,carrera_06,6,2026-10-25,26,26,25,18,643,626,650,0,0,True


In [7]:
for cid in carreras["carrera_id"]:
    df = pd.read_csv(DIR_INTERIM / f"{cid}.csv", dtype=str)
    assert list(df.columns) == COLUMNAS
    print(f"{cid}: {len(df)} filas, {df.shape[1]} columnas, vacíos por columna:",
          df.isna().sum().to_dict())

print()
print("Total de filas:", comprobacion["filas"].sum())
print("¿Cuadran todas?:", comprobacion["cuadra"].all())

carrera_01: 1489 filas, 7 columnas, vacíos por columna: {'posto': 0, 'dorsal': 0, 'nome': 0, 'apelidos': 0, 'posto_sexo': 0, 'posto_categoria': 0, 'tempo': 0}
carrera_02: 929 filas, 7 columnas, vacíos por columna: {'posto': 0, 'dorsal': 0, 'nome': 0, 'apelidos': 0, 'posto_sexo': 0, 'posto_categoria': 0, 'tempo': 0}
carrera_03: 795 filas, 7 columnas, vacíos por columna: {'posto': 0, 'dorsal': 0, 'nome': 0, 'apelidos': 0, 'posto_sexo': 0, 'posto_categoria': 0, 'tempo': 0}
carrera_04: 875 filas, 7 columnas, vacíos por columna: {'posto': 0, 'dorsal': 0, 'nome': 0, 'apelidos': 0, 'posto_sexo': 0, 'posto_categoria': 0, 'tempo': 0}
carrera_05: 659 filas, 7 columnas, vacíos por columna: {'posto': 0, 'dorsal': 0, 'nome': 0, 'apelidos': 0, 'posto_sexo': 0, 'posto_categoria': 0, 'tempo': 0}
carrera_06: 643 filas, 7 columnas, vacíos por columna: {'posto': 0, 'dorsal': 0, 'nome': 0, 'apelidos': 0, 'posto_sexo': 0, 'posto_categoria': 0, 'tempo': 0}

Total de filas: 5390
¿Cuadran todas?: True


## Conclusión

- `descubrir_carreras()` devuelve las carreras ya disputadas con la URL de su Absoluta y
  excluye la última (aún no corrida).
- Todas las páginas están en `datos/raw/<carrera_id>/` y una segunda ejecución no hace
  peticiones nuevas.
- Hay un CSV por carrera en `datos/interim/` y la comprobación de filas está en
  `docs/comprobacion_filas.csv`.